# How to Add a CRS to an `xarray.Dataset` or `xarray.DataArray`

## Problem

This guide shows you how to add a _Coordinate Reference System_ (CRS) to an `xarray.Dataset` or `xarray.DataArray`.

Data granules need to have a CRS specified in a machine readable format to make them interoperable.  This allow tools to perform coordinate transformations.  For many datasets, file contain this information.  However, for some files this information is missing.

See [What is a CRS](TBD) for to learn more about _Coordinate Reference Systems_

## Solution

### Packages

We will use `pyproj` and `xarray` to create a CF-compliant CRS and write it to a [_grid mapping variable_](https://cfconventions.org/Data/cf-conventions/cf-conventions-1.13/cf-conventions.html#grid-mappings-and-projections).

In [1]:
import xarray as xr

from pyproj import CRS

### Load data

We'll use a sea ice concentration data file with CRS information removed from the Near-Real-Time DMSP SSMIS Daily Polar Gridded Sea Ice Concentrations, Version 2 (NSIDC-0081).

In [2]:
ds = xr.open_dataset("../example_data/NSIDC0081_SEAICE_PS_N25km_20230627_v2.0.no_crs.nc")
# ds

ds.F18_ICECON.attrs = {k: v for k, v in ds.F18_ICECON.attrs.items() if k != "grid_mapping"}
ds

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 1, y: 448, x: 304)
Coordinates:
  * time        (time) datetime64[ns] 8B 2023-06-27
  * y           (y) float64 4kB 5.838e+06 5.812e+06 ... -5.312e+06 -5.338e+06
  * x           (x) float64 2kB -3.838e+06 -3.812e+06 ... 3.712e+06 3.738e+06
Data variables:
    F18_ICECON  (time, y, x) float64 1MB ...
Attributes: (12/49)
    title:                     Near-Real-Time DMSP SSMIS Daily Polar Gridded ...
    summary:                   This data set provides a Near-Real-Time (NRT) ...
    id:                        10.5067/YTTHO2FJQ97K
    license:                   Access Constraint: These data are freely, open...
    acknowledgment:            These data are produced and supported by the N...
    metadata_link:             https://doi.org/10.5067/YTTHO2FJQ97K
    ...                        ...
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east
    product_version:           v2.0
    source:                    NOAA Comprehensive Large Array-data Stewardshi...
    instrument:                SSMIS > Special Sensor Microwave Imager/Sounde...
    platform:                  DMSP 5D-3/F16 > Defense Meteorological Satelli...

### Finding the CRS for a dataset

The _Coordinate Reference System_ details for NSIDC datasets can be found in the User Guide's Projection and Grid Description section.  This is usually a table.  If an EPSG code is provided, this is all you need to know to define the projection.  

For the dataset we use as an example here, the EPSG code is 3411.

You can also use the Proj4 string.  For northern hemisphere data for NSIDC-0081 the Proj4 string is
```
+proj=stere +lat_0=90 +lat_ts=70 +lon_0=-45 +k=1 +x_0=0 +y_0=0 +a=6378273 +b=6356889.449 +units=m +no_defs
```

If neither of these CRS definitions are available you will have to define the CRS object using information in the table.  See [how-to-define-a-crs-from-scratch.ipynb](TBD) to learn how to do this.

### Create a `pyproj.CRS` object

To define the `CRS` using an EPSG code, we use the `from_epsg` method.

In [3]:
crs = CRS.from_epsg(3413)

The `crs` object you have just created has a `to_cf` method that returns a Python dictionary that can be used to create a `grid_mapping_variable` for the `xarray.Dataset`.

In [4]:
crs.to_cf()

{'crs_wkt': 'PROJCRS["WGS 84 / NSIDC Sea Ice Polar Stereographic North",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["US NSIDC Sea Ice polar stereographic north",METHOD["Polar Stereographic (variant B)",ID["EPSG",9829]],PARAMETER["Latitude of standard parallel",70,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8832]],PARAMETER["Longitude of origin",-45,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8833]],PARAMETER["False easting",0,

### Create a `grid_mapping_variable`

To create a `grid_mapping_variable` for the `xarray.Dataset` we first create an empty coordinate variable called "crs" using `None`.  The CRS definition is then added to the variable attributes.

In [5]:
ds = ds.assign_coords({"crs": None})
ds.crs.attrs = crs.to_cf()
ds

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 1, y: 448, x: 304)
Coordinates:
  * time        (time) datetime64[ns] 8B 2023-06-27
  * y           (y) float64 4kB 5.838e+06 5.812e+06 ... -5.312e+06 -5.338e+06
  * x           (x) float64 2kB -3.838e+06 -3.812e+06 ... 3.712e+06 3.738e+06
    crs         object 8B None
Data variables:
    F18_ICECON  (time, y, x) float64 1MB ...
Attributes: (12/49)
    title:                     Near-Real-Time DMSP SSMIS Daily Polar Gridded ...
    summary:                   This data set provides a Near-Real-Time (NRT) ...
    id:                        10.5067/YTTHO2FJQ97K
    license:                   Access Constraint: These data are freely, open...
    acknowledgment:            These data are produced and supported by the N...
    metadata_link:             https://doi.org/10.5067/YTTHO2FJQ97K
    ...                        ...
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east
    product_version:           v2.0
    source:                    NOAA Comprehensive Large Array-data Stewardshi...
    instrument:                SSMIS > Special Sensor Microwave Imager/Sounde...
    platform:                  DMSP 5D-3/F16 > Defense Meteorological Satelli...

We then add a `grid_mapping_variable` attribute containing the name of the grid mapping variable we just created to the variables in the dataset.

In [17]:
ds.F18_ICECON.attrs["grid_mapping_variable"] = "crs"
ds

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 1, y: 448, x: 304)
Coordinates:
  * time        (time) datetime64[ns] 8B 2023-06-27
  * y           (y) float64 4kB 5.838e+06 5.812e+06 ... -5.312e+06 -5.338e+06
  * x           (x) float64 2kB -3.838e+06 -3.812e+06 ... 3.712e+06 3.738e+06
    crs         object 8B None
Data variables:
    F18_ICECON  (time, y, x) float64 1MB ...
Attributes: (12/49)
    title:                     Near-Real-Time DMSP SSMIS Daily Polar Gridded ...
    summary:                   This data set provides a Near-Real-Time (NRT) ...
    id:                        10.5067/YTTHO2FJQ97K
    license:                   Access Constraint: These data are freely, open...
    acknowledgment:            These data are produced and supported by the N...
    metadata_link:             https://doi.org/10.5067/YTTHO2FJQ97K
    ...                        ...
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east
    product_version:           v2.0
    source:                    NOAA Comprehensive Large Array-data Stewardshi...
    instrument:                SSMIS > Special Sensor Microwave Imager/Sounde...
    platform:                  DMSP 5D-3/F16 > Defense Meteorological Satelli...

In [6]:
ds.to_netcdf("test.nc")

In [7]:
xr.open_dataset("test.nc")

<xarray.Dataset> Size: 1MB
Dimensions:     (time: 1, y: 448, x: 304)
Coordinates:
  * time        (time) datetime64[ns] 8B 2023-06-27
  * y           (y) float64 4kB 5.838e+06 5.812e+06 ... -5.312e+06 -5.338e+06
  * x           (x) float64 2kB -3.838e+06 -3.812e+06 ... 3.712e+06 3.738e+06
    crs         float64 8B ...
Data variables:
    F18_ICECON  (time, y, x) float64 1MB ...
Attributes: (12/49)
    title:                     Near-Real-Time DMSP SSMIS Daily Polar Gridded ...
    summary:                   This data set provides a Near-Real-Time (NRT) ...
    id:                        10.5067/YTTHO2FJQ97K
    license:                   Access Constraint: These data are freely, open...
    acknowledgment:            These data are produced and supported by the N...
    metadata_link:             https://doi.org/10.5067/YTTHO2FJQ97K
    ...                        ...
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east
    product_version:           v2.0
    source:                    NOAA Comprehensive Large Array-data Stewardshi...
    instrument:                SSMIS > Special Sensor Microwave Imager/Sounde...
    platform:                  DMSP 5D-3/F16 > Defense Meteorological Satelli...